# Analyse des déterminants de l'équipement en Véhicules Électriques

## 1. Introduction

Ce projet vise à comprendre quels facteurs (socio-économiques, infrastructures existantes) influencent le taux d'équipement en véhicules électriques au niveau communal en France.

Nous fusionnons trois sources de données provenant de data.gouv.fr :
- IRVE : Données sur les infrastructures de recharge.
- INSEE : Revenus et données socio-économiques.
- Immatriculations : Parc de VE par commune.

## 2. Chargement et Exploration des Données Brutes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loading import load_all_datasets, charger_communes
from src.cleaning import imputer_valeurs_manquantes_fusion, reparer_encodage_implantation, regrouper_implantation_station, standardize_all_codes, clean_irve_variables_finales, corriger_codes_incoherents, corriger_par_nom, corriger_conflit_code_postal, garder_derniere_observation_commune
from src.features import creer_features_irve, creer_taux_equipement_ve
from src.modelisation import modelisation_lasso, modelisation_xgboost
from src.utils import creer_gdf_irve, joindre_communes, ajouter_codes_geo

# Configuration des chemins
PATHS = {
    'irve': 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435',
    'revenu': 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',
    've': 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
}

# Chargement
df_irve_raw, df_revenu_raw, df_ve_raw = load_all_datasets(PATHS)

#### IRVE

In [ ]:
print(f"Shape : {df_irve_raw.shape}")
df_irve_raw.sample(5)

Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).

In [ ]:
print(f"Le dataset contient {df_irve_raw.shape[0]} lignes pour {df_irve_raw.shape[1]} variables.")

#### Véhicules

In [ ]:
print(f"Shape : {df_ve_raw.shape}")
df_ve_raw.sample(5)

Ce second jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge. Il contient l'historique des immatriculations pour chaque commune à différentes dates.

In [ ]:
print(f"Le dataset contient {df_ve_raw.shape[0]} lignes pour {df_ve_raw.shape[1]} variables.")

#### Revenus

In [ ]:
print(f"Shape : {df_revenu_raw.shape}")
df_revenu_raw.sample(5)

Ce dernier jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.

In [ ]:
print(f"Le dataset contient {df_revenu_raw.shape[0]} lignes pour {df_revenu_raw.shape[1]} variables.")

## 3. Nettoyage et Standardisation Géographique

Le défi majeur de ce projet est la fusion des données. 

#### Choix de la variable de jointure
Les 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus
La jointure sera faite sur ces variables.

#### Préparation des variables de jointure
Cependant les codes INSEE des communes sont souvent mal formatés.

Le diagnostic met en évidence plusieurs points d’attention :
- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

#### Unification des codes
Nous utilisons la fonction standardize_all_codes pour garantir que chaque base dispose d'une colonne code_geo homogène sur 5 caractères.

In [ ]:
dfs_to_clean = {
    "irve": (df_irve_raw, "code_insee_commune"),
    "revenu": (df_revenu_raw, "Code géographique"),
    "ve": (df_ve_raw, "CODGEO")
}

cleaned = standardize_all_codes(dfs_to_clean)
df_irve = cleaned["irve"]
df_revenu = cleaned["revenu"]
df_ve = cleaned["ve"]

Une partie des données IRVE ne possède pas de code INSEE commune, mais dispose de coordonnées latitude/longitude. Plutôt que de supprimer ces lignes, nous utilisons un référentiel géographique pour retrouver la commune correspondante.
Grâce au croisement spatial avec les données de Cartiflette, nous avons pu attribuer un code commune à la quasi-totalité des points de charge.

In [ ]:
# 1. Téléchargement des contours de communes via Cartiflette
communes_fr = charger_communes()

# 2. Transformation de l'IRVE en données géographiques (GeoDataFrame)
gdf_irve = creer_gdf_irve(df_irve, long_col="consolidated_longitude", lat_col="consolidated_latitude")

# 3. Jointure spatiale pour identifier la commune par les coordonnées GPS
gdf_result = joindre_communes(gdf_irve, communes_fr)

# 4. Intégration du code récupéré (code_geo_total) dans le dataframe principal
df_irve = ajouter_codes_geo(df_irve, gdf_result)

Nous avons maintenant 2 colonnes concernant le code géographique dans la base de données IRVE :
- `code_insee_commune` : la variable initiale
- `code_geo_total` : la variable recalculant tous les codes géographiques à partir de la latitude et la longitude

In [ ]:
colonnes_irve = ["code_insee_commune", "code_geo_total"]

print("Valeurs manquantes :")
print({col: df_irve[col].isna().sum() for col in colonnes_irve})

print("\nValeurs uniques :")
print({col: df_irve[col].nunique() for col in colonnes_irve})

Cette manipulation permet de passer de 57919 valeurs manquantes à 1768. De plus les codes recalculés sont plus fiables que ceux initiaux. En effet, les données entrées n'étant pas toujours vérifiées, des erreurs étaient présentes. Certaines code géographiques correspondaient au code postal par exemple.

## 4. Feature Engineering : Passage à l'échelle communale

Réfléchissons aux variables à garder... Lesquelles pourraient être intéressante ?

In [ ]:
list(df_irve.columns)

In [ ]:
list(df_ve.columns)

In [ ]:
list(df_revenu.columns)

Nos bases n'ont pas la même unité d'observation. L'IRVE liste des bornes (plusieurs par ville), alors que nous voulons une analyse par commune.

Pour pouvoir efectuer la jointure sur les codes géographiques, il est essentiel que pour chacune de nos 3 bases de données, chaque ligne corresponde à un unique code géographique.

Il faut alors réfléchir à la façon dont on veut agréger df_irve et quelles variables on souhaite garder.

Il faut également étudier la base df_ve afin de supprimer les "doublons" de code géographique.

#### IRVE
Ici, l'objectif est de mesurer l'offre de recharge par commune. Comme il peut y avoir plusieurs points de recharge par commune, il faut agréger. L'objectif final est d'expliquer ou prédire le taux de véhicules électriques local.

Après étude de notre base de données (cf etude_df_irve.ipynb) nous avons sélectionnés plusieurs variables nous permettant d'en créer de nouvelles, agrégées par code géographique.

Certaines variables ont été écarté directement car jugées non pertinente pour notre sujet, d'autres présentaient trop de valeurs manquantes, d'autres étaient trop peu informatives ou encore était du texte de libre donc trop difficile à utiliser avec le peu de temps dont nous disposons.

Fianleamnt voici un récap de la création des variables agrégées :

| Variables utilisées       | Variable finale                              | Construction        |
|--------------------------|-----------------------------------------------|---------------------|
| nbre_pdc                 | total_pdc                                     | somme               |
| puissance_nominale       | puissance_moyenne                             | moyenne             |
| puissance_nominale       | puissance_max                                 | max                 |
| nom_operateur            | nb_operateurs                                 | nunique             |
| nom_operateur            | top_operateur                                 | mode                |
| prise_type_2             | pct_type_2                                    | moyenne booléenne   |
| prise_type_combo_ccs     | pct_combo_ccs                                 | moyenne booléenne   |
| prise_type_chademo       | pct_chademo                                   | moyenne booléenne   |
| prise_type_ef            | pct_type_ef                                   | moyenne booléenne   |
| gratuit                  | pct_gratuit                                   | moyenne booléenne   |
| paiement_cb              | pct_paiement_cb                               | moyenne booléenne   |
| paiement_autre           | pct_paiement_autre                            | moyenne booléenne   |
| implantation_station     | prive, public, rapide, voirie                 | dummies + moyenne   |

Commençons par rendre les données des variables sélectionnées propres avant de faire l'agrégation.

En particulier, les données d'implantation des stations présentent des erreurs d'encodage (caractères spéciaux corrompus) et une trop grande diversité de labels. Nous procédons à une réparation du texte puis à un regroupement en 4 catégories majeures : privé, public, voirie et rapide.

In [ ]:
# Traitement IRVE : Transformation des types et agrégation
df_irve = reparer_encodage_implantation(df_irve)
df_irve = regrouper_implantation_station(df_irve)
df_irve_clean = clean_irve_variables_finales(df_irve)
df_irve_final = creer_features_irve(df_irve_clean)
df_irve_final.sample(5)

#### Véhicules
La base véhicules contient plusieurs dates d'observation pour une même commune.

Afin d'être cohérent avec les autres sources de données (IRVE, INSEE), nous souhaitons disposer d'une seule ligne par commune correspondant à la situation la plus récente disponible.

Nous conservons donc, pour chaque code commune (`CODGEO`), la dernière observation temporelle.

In [ ]:
# Traitement VE : Sélection de la dernière observation
df_ve_final = garder_derniere_observation_commune(df_ve)
df_ve_final.head()

À partir de cette base de données, nous allons construire la **variable cible** de notre analyse : le **taux de véhicules électriques par commune**.

Cette variable correspond à un ratio calculé à partir des variables suivantes :

- `NB_VP` : nombre total de voitures particulières ;
- `NB_VP_RECHARGEABLES_EL` : nombre de voitures particulières rechargeables / électriques.

Le taux est défini par la formule suivante :

$$
taux\_equipement\_ve = \frac{NB\_VP\_RECHARGEABLES\_EL}{NB\_VP}
$$

Ce ratio est plus pertinent que le nombre brut de véhicules électriques, car il permet de **neutraliser l’effet de taille des communes**.

En effet, une grande commune comptera naturellement davantage de véhicules électriques qu’une petite commune. Le recours à un taux permet donc de comparer plus justement le niveau d’équipement entre communes, quelle que soit leur population ou leur parc automobile total.

In [ ]:
df_ve_final = creer_taux_equipement_ve(df_ve_final)
df_ve_final["taux_equipement_ve"].describe().round(4)

Nous sélectionnons pour la modélisation les variables suivantes :
- CODGEO
- NB_VP
- NB_VP_RECHARGEABLES_EL
- taux_equipement_ve

In [ ]:
vars_finales = [
        "CODGEO",
        "NB_VP",
        "NB_VP_RECHARGEABLES_EL",
        "taux_equipement_ve"
    ]
df_ve_final = df_ve_final[vars_finales]
df_ve_final.head(5)

#### Revenus

Nous sélectionnons pour la modélisation les variables suivantes :
- Code géographique
- Libellé géographique
- [DISP] Médiane (€)
- [DISP] Iice de Gini
- [DISP] Nbre de ménages fiscaux
- [DISP] Nbre de personnes dans les ménages fiscaux
- [DISP] Part des revenus d’activité (%)
- [DISP] dont part des revenus des activités non salariées (%)
- [DISP] Part des revenus du patrimoine et autres revenus (%)

Pour plus de détails, veuillez vous référer au notebook « etude_df_revenu ».

In [ ]:
var_selec = ['Code géographique',
'Libellé géographique',
'[DISP] Médiane (€)',
 '[DISP] Iice de Gini',
 '[DISP] Nbre de ménages fiscaux',
 '[DISP] Nbre de personnes dans les ménages fiscaux',
 '[DISP] Part des revenus d’activité (%)',
 '[DISP] dont part des revenus des activités non salariées (%)',
 '[DISP] Part des revenus du patrimoine et autres revenus (%)'
]
df_revenu_final=df_revenu[var_selec]
df_revenu_final.sample(5)

## 5. Jointure

Pour la base de données jointe nous avons choisis de garder les variables suivantes :
- IRVE
    - total_pdc
    - puissance_moyenne
    - puissance_max
    - nb_operateurs
    - top_operateur
    - pct_type_2
    - pct_combo_ccs
    - pct_chademo
    - pct_type_ef
    - pct_gratuit
    - pct_paiement_cb
    - pct_paiement_autre
    - prive
    - public
    - rapide
    - voirie          
- Véhicules
    - CODGEO
    - NB_VP
    - NB_VP_RECHARGEABLES_EL
    - taux_equipement_ve
- Revenus
    - Code géographique
    - Libellé géographique
    - [DISP] Médiane (€)
    - [DISP] Iice de Gini
    - [DISP] Nbre de ménages fiscaux
    - [DISP] Nbre de personnes dans les ménages fiscaux
    - [DISP] Part des revenus d’activité (%)
    - [DISP] dont part des revenus des activités non salariées (%)
    - [DISP] Part des revenus du patrimoine et autres revenus (%)

In [ ]:
# Fusion des trois bases
df_final = (
    df_ve_final
    .merge(df_irve_final, left_on="CODGEO", right_on="code_geo_total", how="left")
    .merge(df_revenu_final, left_on="CODGEO", right_on="Code géographique", how="left")
)

print("Shape base finale :", df_final.shape)
df_final.sample(5)

## 6. Traitement des valeurs manquantes

In [ ]:
df_final.isna().sum()

La base de données finale contient **35 197 communes**.

Parmi celles-ci, **24 919 communes ne disposent d’aucune donnée issue de la base IRVE**, soit environ **70 % de l’échantillon total**.

---

Après analyse du contexte territorial français, nous faisons l’hypothèse que ces communes peuvent être considérées comme **non équipées en points de recharge (IRVE)**.

Cette hypothèse est jugée réaliste dans le contexte suivant :

- faible densité de population ;
- activité économique limitée ;
- faible présence de parkings publics ;
- flux routiers réduits.

Dans ce type de communes, l’absence d’infrastructures de recharge est très fréquente et cohérente avec la structure territoriale française.

---

Les bornes de recharge sont principalement concentrées dans les zones suivantes :

- grandes villes et agglomérations ;
- axes routiers majeurs ;
- zones commerciales ;
- parkings publics structurants ;
- pôles intercommunaux centraux.

Ainsi, de nombreuses petites communes rurales ou périphériques ne disposent d’aucune infrastructure de recharge sur leur territoire.

---

Il est essentiel de noter qu’une absence locale de borne ne signifie pas nécessairement une absence d’accès à la recharge.
En effet une commune peut être dépourvue d’IRVE tout en étant située à quelques kilomètres d’une commune équipée, ce qui permet un accès indirect à la recharge.

---

L’absence de données IRVE pour une commune est donc interprétée comme une absence d’infrastructure locale

In [ ]:
df_imputation = df_final.copy()

COLS_IRVE_NUMERIQUES = [
    'total_pdc', 
    'puissance_moyenne', 
    'puissance_max', 
    'nb_operateurs', 
    'pct_type_2', 
    'pct_combo_ccs', 
    'pct_chademo', 
    'pct_type_ef', 
    'pct_gratuit', 
    'pct_paiement_cb', 
    'pct_paiement_autre',
    'prive',
    'public',
    'rapide',
    'voirie'
]

COLS_IRVE_TEXTE = ['top_operateur']

df_imputation = imputer_valeurs_manquantes_fusion(df_imputation, cols_to_zero= COLS_IRVE_NUMERIQUES, cols_to_label=COLS_IRVE_TEXTE)
df_imputation.sample(5)

Nous pouvons désormais supprimer les variables `code_geo_total` et `Code géographique`.

Ces deux colonnes apportent une information redondante par rapport à `CODGEO`. Afin de simplifier la structure du jeu de données et d’éviter toute duplication inutile d’information, nous conservons uniquement la variable `CODGEO`, qui constitue l’identifiant géographique de référence.

In [ ]:
df_imputation = df_imputation.drop(columns=["code_geo_total", "Code géographique"])

------------------------ début -------------

In [ ]:
missing = pd.DataFrame({
    "nb_manquants": df_imputation.isna().sum(),
    "pct_manquants": (df_imputation.isna().mean() * 100).round(2)
})

missing.sort_values("nb_manquants", ascending=False)

[DISP] Iice de Gini
[DISP] dont part des revenus des activités non salariées (%)
[DISP] Part des revenus du patrimoine et autres revenus (%)
[DISP] Part des revenus d’activité (%)

Il va falloir enlever ces variables... trop de valeurs manquantes !

------------------------ fin -------------

## 7. Analyse des variables

A faire :
- analyse de la variable cible
- analyse descriptive des variables explicatives (univariées + bivariées)
- étude des corrélations
- nouveau tri des variables à la suite des analyses
- vérifier que le format des variables est correcte pour la modélisation.

## 8. Modélisation

Nous cherchons à prédire la variable taux_equipement_ve.

Approche 1 : Régression Lasso
Le modèle Lasso est idéal pour l'interprétabilité. Il permet de voir quelles variables "pèsent" réellement dans la décision d'achat d'un véhicule électrique.

Approche 2 : XGBoost
Ce modèle non-linéaire permet ...

Nos variables pour la modélisation sont les suivantes :

attention LASSO ne prend pas de valeurs manquantes !!

In [ ]:
features = ['total_pdc', 'puissance_max', 'nb_operateurs', 'pct_gratuit']
target = 'taux_equipement_ve' 

df_model = df_final[features + [target]].dropna()

# 2. Exécuter le Lasso (Baseline & Sélection)
model_l, metrics_l, coeffs_l = modelisation_lasso(df_model, features, target)
print(f"R2 Lasso : {metrics_l['r2']:.3f}")
print("Déterminants (Lasso) :\n", coeffs_l[coeffs_l != 0])

# 3. Exécuter le Gradient Boosting (Performance)
model_x, metrics_x, importance_x = modelisation_xgboost(df_final, features, target)
print(f"R2 XGBoost : {metrics_x['r2']:.3f}")
importance_x.plot(kind='barh', title="Importance des variables - XGBoost")

In [ ]:
features = ['[DISP] Médiane (€)', '[DISP] Iice de Gini', 'total_pdc', 'puissance_moyenne']
target = 'taux_equipement_ve'

model_lasso, metrics_lasso, coeffs = modelisation_lasso(df_master, features, target)
model_xgb, metrics_xgb, importance = modelisation_xgboost(df_master, features, target)

## 9. Analyse des résultats